In [4]:
# Importer les bibliothèques necessaires
!pip install gdown
import gdown
import pandas as pd
import re
import gc
import os
import sys
import pandas as pd
import numpy as np
import tqdm
import seaborn as sns

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE
from scipy.stats import chi2_contingency
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix,
    precision_score, recall_score, f1_score, accuracy_score
)
from sklearn.metrics import f1_score


from sklearn.svm import SVC

In [5]:
np.random.seed(123)

In [6]:
# Charger les données
df = pd.read_csv('Fraud Detection Dataset.csv')  # changer le nom du fichier si nécessaire
print('Forme du dataset chargé :', df.shape)
print(df['Fraudulent'].value_counts())                                                                   

Forme du dataset chargé : (51000, 12)
Fraudulent
0    48490
1     2510
Name: count, dtype: int64


In [7]:
TARGET = "Fraudulent"

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df[TARGET],
    random_state=42
)


# **Sommaire**

### Traitement des valeurs manquantes Na
1. 1ère méthode : moyenne
2. 2ème méthode : Interpolation Bayesienne
3. 3ième méthode : MissForest

### Implémentation des modèles de classification
1. *RandomForest*
2. *Régression Logistique*
3. *SVM*
4. *LightGBM*

### Algorithme de génération de données synthétiques : SMOTE

0. Implémentation SMOTE

### Résultats **sans SMOTE**
1. Mesures d'évaluation des modèles
2. Comparaison des courbes ROC et AUC


### Résultats **avec SMOTE**
1. Mesures d'évaluation des modèles
2. Comparaison des courbes ROC et AUC



## **Traitement des valeurs manquantes Na**


### **1ère méthode : moyenne**

Implémentation fonction *missing_mean*

In [8]:

def missing_mean(df, target_col="Fraudulent"):
    """
    Imputation par la moyenne (numérique) et le mode (catégoriel).
    La variable cible n'est JAMAIS utilisée pour l'imputation.
    """

    df_c = df.copy()

    # Colonnes numériques (hors cible)
    numeric_cols = (
        df_c.select_dtypes(include=['int64', 'float64'])
        .columns
        .drop(target_col)
    )

    # Colonnes catégorielles
    categorical_cols = df_c.select_dtypes(
        include=['object', 'category']
    ).columns

    # Numériques → moyenne globale
    for col in numeric_cols:
        mean_value = df_c[col].mean()
        df_c[col] = df_c[col].fillna(mean_value)

    # Catégorielles → mode global
    for col in categorical_cols:
        if df_c[col].isna().any():
            mode_series = df_c[col].mode()
            if not mode_series.empty:
                df_c[col] = df_c[col].fillna(mode_series.iloc[0])

    # Diagnostic
    print("Pourcentage de valeurs manquantes après imputation (Mean) :")
    print((df_c.isnull().mean() * 100).round(3))

    return df_c

    # SPLIT
    # SPLIT (une seule fois, commun à toutes les méthodes)
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Fraudulent"]
)

# Imputation par la moyenne
df_train_mean = missing_mean(df_train)
df_test_mean  = missing_mean(df_test)



Pourcentage de valeurs manquantes après imputation (Mean) :
Transaction_ID                      0.0
User_ID                             0.0
Transaction_Amount                  0.0
Transaction_Type                    0.0
Time_of_Transaction                 0.0
Device_Used                         0.0
Location                            0.0
Previous_Fraudulent_Transactions    0.0
Account_Age                         0.0
Number_of_Transactions_Last_24H     0.0
Payment_Method                      0.0
Fraudulent                          0.0
dtype: float64
Pourcentage de valeurs manquantes après imputation (Mean) :
Transaction_ID                      0.0
User_ID                             0.0
Transaction_Amount                  0.0
Transaction_Type                    0.0
Time_of_Transaction                 0.0
Device_Used                         0.0
Location                            0.0
Previous_Fraudulent_Transactions    0.0
Account_Age                         0.0
Number_of_Transactions_La

### **2ème méthode : Interpolation Bayesienne**

Implémentation fonction *missing_mean*

In [9]:
def bayesian_numeric_imputer_fit(
    df_train,
    target_col,
    num_cols=None,
    alpha=1.0,
    random_state=None
):
    """
    Apprend les paramètres bayésiens (posterior Ridge)
    à partir du TRAIN uniquement.
    """
    if random_state is not None:
        np.random.seed(random_state)

    df = df_train.copy()

    # Colonnes numériques (hors target)
    if num_cols is None:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        num_cols = [c for c in num_cols if c != target_col]

    models = {}

    for col in num_cols:
        if df[col].isna().sum() == 0:
            continue

        mask_obs = df[col].notna()
        y = df.loc[mask_obs, col].values

        X = df.loc[mask_obs, num_cols].copy()
        X = X.fillna(X.mean())

        X_mat = X.values

        XtX = X_mat.T @ X_mat
        XtY = X_mat.T @ y

        Sigma_beta = np.linalg.inv(XtX + alpha * np.eye(X_mat.shape[1]))
        beta_hat = Sigma_beta @ XtY

        residuals = y - X_mat @ beta_hat
        sigma2 = residuals.var()

        models[col] = {
            "beta": beta_hat,
            "Sigma_beta": Sigma_beta,
            "sigma2": sigma2,
            "predictors": num_cols,
            "mean_X": X.mean()
        }

    return models


# Apllication de l'imputation numérique (TRAIN OU TEST)

def bayesian_numeric_imputer_transform(df, models, random_state=None):
    """
    Applique l'imputation bayésienne à partir des paramètres appris.
    """
    if random_state is not None:
        np.random.seed(random_state)

    df_out = df.copy()

    for col, model in models.items():
        mask_miss = df_out[col].isna()
        if mask_miss.sum() == 0:
            continue

        X_miss = df_out.loc[mask_miss, model["predictors"]]
        X_miss = X_miss.fillna(model["mean_X"])
        Xm = X_miss.values

        pred_mean = Xm @ model["beta"]
        pred_var = model["sigma2"] + np.sum(
            Xm @ model["Sigma_beta"] * Xm, axis=1
        )

        draws = np.random.normal(pred_mean, np.sqrt(pred_var))
        df_out.loc[mask_miss, col] = draws

    return df_out


# Imputation catégorielle DIRICHLET

def categorical_dirichlet_fit(df_train, cat_cols, alpha0=1.0):
    """
    Apprend les postérieurs Dirichlet sur le TRAIN.
    """
    models = {}

    for col in cat_cols:
        observed = df_train[col].dropna().astype(str)
        counts = observed.value_counts()

        alpha_prior = np.ones(len(counts)) * alpha0
        posterior = counts.values + alpha_prior
        probs = posterior / posterior.sum()

        models[col] = {
            "categories": counts.index.tolist(),
            "probs": probs
        }

    return models


# Application catégorielle  
def categorical_dirichlet_transform(df, models, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)

    df_out = df.copy()

    for col, model in models.items():
        mask_miss = df_out[col].isna()
        n = mask_miss.sum()

        if n == 0:
            continue

        draws = np.random.choice(
            model["categories"],
            size=n,
            p=model["probs"]
        )

        df_out.loc[mask_miss, col] = draws

    return df_out

# Pipeline Final


# CONFIGURATION GLOBALE

TARGET = "Fraudulent"
RANDOM_STATE = 42

# Colonnes catégorielles à imputer (à adapter si besoin)
CAT_COLS = ["Transaction_Type","Device_Used", "Location", "Payment_Method"]


# SPLIT

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[TARGET]
)

# FIT (SUR LE TRAIN)

num_models = bayesian_numeric_imputer_fit(
    df_train,
    target_col=TARGET,
    random_state=42
)

cat_models = categorical_dirichlet_fit(
    df_train,
    CAT_COLS
)


# TRANSFORM


df_train_bayes = bayesian_numeric_imputer_transform(
    df_train, num_models, random_state=42
)
df_train_bayes = categorical_dirichlet_transform(
    df_train_bayes, cat_models, random_state=42
)

df_test_bayes = bayesian_numeric_imputer_transform(
    df_test, num_models, random_state=42
)
df_test_bayes = categorical_dirichlet_transform(
    df_test_bayes, cat_models, random_state=42
)



In [10]:
# Vérification du fonctionnement de la méthode Bayésienne
print("NaN avant (train) :")
print(df_train.isna().sum())

print("\nNaN après (train) :")
print(df_train_bayes.isna().sum())

NaN avant (train) :
Transaction_ID                         0
User_ID                                0
Transaction_Amount                  2036
Transaction_Type                       0
Time_of_Transaction                 2039
Device_Used                         1972
Location                            2006
Previous_Fraudulent_Transactions       0
Account_Age                            0
Number_of_Transactions_Last_24H        0
Payment_Method                      1967
Fraudulent                             0
dtype: int64

NaN après (train) :
Transaction_ID                      0
User_ID                             0
Transaction_Amount                  0
Transaction_Type                    0
Time_of_Transaction                 0
Device_Used                         0
Location                            0
Previous_Fraudulent_Transactions    0
Account_Age                         0
Number_of_Transactions_Last_24H     0
Payment_Method                      0
Fraudulent                          

### **3ème méthode : MissForest**

In [11]:
def missforest_impute(
    df,
    target_col,
    n_estimators=100,
    max_iter=5,
    random_state=42
):
    """
    Imputation MissForest (approximation académique).
    La variable cible n'est JAMAIS imputée.
    """

    df_imp = df.copy()

    # Séparer types de colonnes
    num_cols = df_imp.select_dtypes(include=['number']).columns.tolist()
    cat_cols = df_imp.select_dtypes(include=['object','category']).columns.tolist()

    # Exclure la cible
    if target_col in num_cols:
        num_cols.remove(target_col)
    if target_col in cat_cols:
        cat_cols.remove(target_col)

    # Colonnes à imputer (hors cible)
    cols_with_na = [
        col for col in df_imp.columns
        if df_imp[col].isna().any() and col != target_col
    ]

    # === Initialisation simple ===
    for col in num_cols:
        df_imp[col] = df_imp[col].fillna(df_imp[col].mean())

    for col in cat_cols:
        df_imp[col] = df_imp[col].fillna(df_imp[col].mode()[0])

    print("Initialisation de MissForest terminée.")

    # === Boucle MissForest ===
    for it in range(max_iter):
        print(f"\n--- Itération {it+1}/{max_iter} ---")

        for col in cols_with_na:
            print(f"Imputation de : {col}")

            mask_obs  = df_imp[col].notna()
            mask_miss = df_imp[col].isna()

            if mask_miss.sum() == 0:
                continue

            # X = autres variables (sans la cible)
            X_train = df_imp.loc[mask_obs].drop(columns=[col, target_col])
            X_pred  = df_imp.loc[mask_miss].drop(columns=[col, target_col])

            y_train = df_imp.loc[mask_obs, col]

            # Encodage catégoriel
            X_train = pd.get_dummies(X_train, drop_first=True)
            X_pred  = pd.get_dummies(X_pred, drop_first=True)

            # Alignement strict
            X_train, X_pred = X_train.align(
                X_pred, join="left", axis=1, fill_value=0
            )

            # Choix du modèle
            if col in num_cols:
                model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    random_state=random_state,
                    n_jobs=-1
                )
            else:
                model = RandomForestClassifier(
                    n_estimators=n_estimators,
                    random_state=random_state,
                    n_jobs=-1
                )

            # Entraînement
            model.fit(X_train, y_train)

            # Prédiction
            y_pred = model.predict(X_pred)

            # Injection
            df_imp.loc[mask_miss, col] = y_pred

    return df_imp

# SPLIT

TARGET = "Fraudulent"
RANDOM_STATE = 42

df = pd.read_csv("Fraud Detection Dataset.csv")

df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[TARGET]
)

# TRAIN

df_train_mf = missforest_impute(
    df_train,
    target_col=TARGET,
    n_estimators=200,
    max_iter=5,
    random_state=RANDOM_STATE
)

# TEST (approximation, à documenter)

df_test_mf = missforest_impute(
    df_test,
    target_col=TARGET,
    n_estimators=200,
    max_iter=5,
    random_state=RANDOM_STATE
)



Initialisation de MissForest terminée.

--- Itération 1/5 ---
Imputation de : Transaction_Amount
Imputation de : Time_of_Transaction
Imputation de : Device_Used
Imputation de : Location
Imputation de : Payment_Method

--- Itération 2/5 ---
Imputation de : Transaction_Amount
Imputation de : Time_of_Transaction
Imputation de : Device_Used
Imputation de : Location
Imputation de : Payment_Method

--- Itération 3/5 ---
Imputation de : Transaction_Amount
Imputation de : Time_of_Transaction
Imputation de : Device_Used
Imputation de : Location
Imputation de : Payment_Method

--- Itération 4/5 ---
Imputation de : Transaction_Amount
Imputation de : Time_of_Transaction
Imputation de : Device_Used
Imputation de : Location
Imputation de : Payment_Method

--- Itération 5/5 ---
Imputation de : Transaction_Amount
Imputation de : Time_of_Transaction
Imputation de : Device_Used
Imputation de : Location
Imputation de : Payment_Method
Initialisation de MissForest terminée.

--- Itération 1/5 ---
Imputatio

In [12]:
# Sauvegaarde
df_train_mf.to_csv("Fraud_Dataset_MissForest_TRAIN.csv", index=False)
df_test_mf.to_csv("Fraud_Dataset_MissForest_TEST.csv", index=False)

print("✔ Imputation MissForest terminée et sauvegardée.")



✔ Imputation MissForest terminée et sauvegardée.


**FONCTION SEPARATION**

In [13]:
def separation(df, target_col="Fraudulent"):
    """
    Sépare un DataFrame en variables explicatives (X) et variable cible (y).

    Paramètres
    ----------
    df : pandas.DataFrame
        Dataset complet (après imputation si nécessaire)
    target_col : str
        Nom de la variable cible (par défaut 'Fraudulent')

    Sortie
    ------
    X : pandas.DataFrame
        Variables explicatives
    y : pandas.Series
        Variable cible
    """

    X = df.drop(columns=[target_col])
    y = df[target_col]

    return X, y


### **Implémentation des modèles de classification**


### 1. *RandomForest*

In [14]:
def random_forest_classifier(
    X_train,
    y_train,
    X_test,
    y_test,
    n_estimators=100,
    random_state=42
):
    """
    Entraîne un Random Forest et évalue ses performances.

    Paramètres
    ----------
    X_train, y_train : données d'entraînement
    X_test, y_test   : données de test

    Sorties
    -------
    y_pred        : prédictions de classes
    y_pred_proba  : probas P(Fraudulent = 1)
    metrics       : dictionnaire de scores
    """

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=random_state,
        n_jobs=-1
    )

    # Entraînement
    clf.fit(X_train, y_train)

    # Prédictions
    y_pred = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]

    # Évaluation
    metrics = {
        "AUC": roc_auc_score(y_test, y_pred_proba),
        "Accuracy": accuracy_score(y_test, y_pred)
    }

    return y_pred, y_pred_proba, metrics




### 2. *Régression Logistique*

Implémentation fonction *logistic*

In [15]:
def logistic_regression(A, b, T):
    """
    Régression logistique binaire.
    
    Entrée :
    - (A, b) : données d'entraînement
    - T      : données de test

    Sortie :
    - y_pred        : prédictions de classes
    - y_pred_proba  : probas P(Fraudulent = 1)
    """

    lr = LogisticRegression(
        max_iter=1000,
        solver="lbfgs"
    )

    # Entraînement
    lr.fit(A, b)

    # Prédictions
    y_pred = lr.predict(T)
    y_pred_proba = lr.predict_proba(T)[:, 1]

    return y_pred, y_pred_proba




### 3. *SVM*

In [16]:
def svm_classifier(A, b, T):
    """
    SVM RBF avec standardisation des variables.

    Entrée :
    - (A, b) : données d'entraînement
    - T      : données de test

    Sortie :
    - y_pred        : prédictions de classes
    - y_pred_proba  : probas P(Fraudulent = 1)
    """

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(
            kernel="rbf",
            C=5,
            gamma="scale",
            probability=True,
            random_state=42
        ))
    ])

    # Entraînement
    pipeline.fit(A, b)

    # Prédictions
    y_pred = pipeline.predict(T)
    y_pred_proba = pipeline.predict_proba(T)[:, 1]

    return y_pred, y_pred_proba

### 4. *LightGBM*

In [17]:
import lightgbm as lgb

In [18]:
# ================================
# Préparation des données pour LightGBM
# ================================

# Étape 1 — Séparation X / y
X_train, y_train = separation(df_train_bayes)
X_test,  y_test  = separation(df_test_bayes)

# Étape 2 — Suppression des identifiants
ID_COLS = ["Transaction_ID", "User_ID"]

X_train = X_train.drop(columns=ID_COLS)
X_test  = X_test.drop(columns=ID_COLS)

# Étape 3 — Détection des variables catégorielles
cat_cols = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Variables catégorielles utilisées :", cat_cols)

# Étape 4 — Conversion en type 'category'
for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col]  = X_test[col].astype("category")


Variables catégorielles utilisées : ['Transaction_Type', 'Device_Used', 'Location', 'Payment_Method']


In [19]:
import lightgbm as lgb

def lightgbm_classifier_full(X_train, y_train, X_test, cat_cols):
    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        min_data_in_leaf=50,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train,
        categorical_feature=cat_cols
    )

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return y_pred, y_proba, model


### **Métriques d'évaluation des modèles et courbes ROC (et AUC)**

1. Implémentation fonction résultat métriques

In [20]:

def evaluate_metrics(y_true, y_proba, y_pred=None, synthese=False):
    """
    Évaluation d'un modèle de classification binaire (fraude).

    Paramètres
    ----------
    y_true : array-like
        Vraies classes (0/1)
    y_proba : array-like
        Probabilité P(Fraudulent = 1)
    y_pred : array-like, optionnel
        Prédictions au seuil 0.5 (si déjà calculées)
    synthese : bool
        Si True, retourne une liste de métriques

    Sortie
    ------
    Métriques affichées ou liste de métriques
    """

    # === ROC & AUC ===
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    auc_score = auc(fpr, tpr)

    # === Indice de Youden ===
    youden = tpr - fpr
    idx = np.argmax(youden)
    threshold_opt = thresholds[idx]

    # Prédictions au seuil optimal
    y_pred_opt = (y_proba >= threshold_opt).astype(int)

    cm = confusion_matrix(y_true, y_pred_opt)
    TN, FP, FN, TP = cm.ravel()

    recall = TP / (TP + FN)
    precision = TP / (TP + FP)
    specificity = TN / (TN + FP)
    f1 = 2 * precision * recall / (precision + recall)

    # === Seuil 0.5 ===
    if y_pred is None:
        y_pred = (y_proba >= 0.5).astype(int)

    acc_05 = accuracy_score(y_true, y_pred)
    prec_05 = precision_score(y_true, y_pred)
    rec_05 = recall_score(y_true, y_pred)
    f1_05 = f1_score(y_true, y_pred)

    if synthese:
        return [
            auc_score,
            threshold_opt,
            recall,
            specificity,
            precision,
            f1,
            acc_05,
            prec_05,
            rec_05,
            f1_05
        ]

    else:
        print("===== Évaluation du modèle =====")
        print(f"AUC : {auc_score:.4f}")
        print(f"Seuil optimal (Youden) : {threshold_opt:.4f}")
        print("--- Au seuil optimal ---")
        print(f"Recall (sensibilité) : {recall:.4f}")
        print(f"Precision : {precision:.4f}")
        print(f"Specificity : {specificity:.4f}")
        print(f"F1-score : {f1:.4f}")

        print("\n--- Au seuil 0.5 ---")
        print(f"Accuracy : {acc_05:.4f}")
        print(f"Recall : {rec_05:.4f}")
        print(f"Precision : {prec_05:.4f}")
        print(f"F1-score : {f1_05:.4f}")


A. LightGBM + Bayésien

In [21]:
# ================================
# LightGBM + Bayésien (SANS SMOTE)
# ================================

# 1) Séparation X / y
X_train_bayes, y_train_bayes = separation(df_train_bayes)
X_test_bayes,  y_test        = separation(df_test_bayes)

# 2) Suppression des identifiants
ID_COLS = ["Transaction_ID", "User_ID"]
X_train_bayes = X_train_bayes.drop(columns=ID_COLS)
X_test_bayes  = X_test_bayes.drop(columns=ID_COLS)

# 3) Détection des variables catégorielles
cat_cols = X_train_bayes.select_dtypes(
    include=["object", "category"]
).columns.tolist()

# 4) Conversion en 'category' (LightGBM natif)
for col in cat_cols:
    X_train_bayes[col] = X_train_bayes[col].astype("category")
    X_test_bayes[col]  = X_test_bayes[col].astype("category")

# 5) Modèle LightGBM (pondération des classes)
y_pred_bayes, y_proba_bayes, model = lightgbm_classifier_full(
    X_train_bayes,
    y_train_bayes,
    X_test_bayes,
    cat_cols
)

# 6) Évaluation
print("\n=== LightGBM + Bayésien (sans SMOTE) ===")
evaluate_metrics(
    y_true=y_test,
    y_proba=y_proba_bayes,
    y_pred=y_pred_bayes,
    synthese=False
)


[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Number of positive: 2008, number of negative: 38792
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000565 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 676
[LightGBM] [Info] Number of data points in the train set: 40800, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be i

In [22]:
print("Nombre de variables utilisées par LightGBM :", model.n_features_)
print("Variables fournies au modèle :", X_train_bayes.shape[1])

Nombre de variables utilisées par LightGBM : 9
Variables fournies au modèle : 9


In [23]:
# ================================
# LightGBM + Mean (SANS SMOTE)
# ================================

# 1) Séparation X / y
X_train_mean, y_train_mean = separation(df_train_mean)
X_test_mean,  y_test       = separation(df_test_mean)

# 2) Suppression des identifiants (jamais informatifs)
ID_COLS = ["Transaction_ID", "User_ID"]
X_train_mean = X_train_mean.drop(columns=ID_COLS)
X_test_mean  = X_test_mean.drop(columns=ID_COLS)

# 3) Détection des variables catégorielles
cat_cols = X_train_mean.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Variables catégorielles utilisées :", cat_cols)

# 4) Conversion en type 'category' (LightGBM natif)
for col in cat_cols:
    X_train_mean[col] = X_train_mean[col].astype("category")
    X_test_mean[col]  = X_test_mean[col].astype("category")

# 5) Entraînement + prédiction (sans SMOTE, avec pondération)
y_pred_mean, y_proba_mean, model_mean = lightgbm_classifier_full(
    X_train_mean,
    y_train_mean,
    X_test_mean,
    cat_cols
)

# 6) Évaluation
print("\n=== LightGBM + Mean (sans SMOTE) ===")
evaluate_metrics(
    y_true=y_test,
    y_proba=y_proba_mean,
    y_pred=y_pred_mean,
    synthese=False
)


Variables catégorielles utilisées : ['Transaction_Type', 'Device_Used', 'Location', 'Payment_Method']
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Number of positive: 2008, number of negative: 38792
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000720 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 446
[LightGBM] [Info] Number of data points in the train set: 40800, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current valu

In [24]:
# ================================
# LightGBM + MissForest (SANS SMOTE)
# ================================

# 1) Séparation X / y
X_train_mf, y_train_mf = separation(df_train_mf)
X_test_mf,  y_test     = separation(df_test_mf)

# 2) Suppression des identifiants
ID_COLS = ["Transaction_ID", "User_ID"]
X_train_mf = X_train_mf.drop(columns=ID_COLS)
X_test_mf  = X_test_mf.drop(columns=ID_COLS)

# 3) Détection des variables catégorielles
cat_cols = X_train_mf.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Variables catégorielles utilisées :", cat_cols)

# 4) Conversion en type 'category' (LightGBM natif)
for col in cat_cols:
    X_train_mf[col] = X_train_mf[col].astype("category")
    X_test_mf[col]  = X_test_mf[col].astype("category")

# 5) Entraînement + prédiction (sans SMOTE, avec pondération)
y_pred_mf, y_proba_mf, model_mf = lightgbm_classifier_full(
    X_train_mf,
    y_train_mf,
    X_test_mf,
    cat_cols
)

# 6) Évaluation
print("\n=== LightGBM + MissForest (sans SMOTE) ===")
evaluate_metrics(
    y_true=y_test,
    y_proba=y_proba_mf,
    y_pred=y_pred_mf,
    synthese=False
)


Variables catégorielles utilisées : ['Transaction_Type', 'Device_Used', 'Location', 'Payment_Method']
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Number of positive: 2008, number of negative: 38792
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000670 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 446
[LightGBM] [Info] Number of data points in the train set: 40800, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current valu

In [25]:
print("Nombre de variables fournies :", X_train_mf.shape[1])
print("Nombre de variables utilisées :", model_mf.n_features_)

Nombre de variables fournies : 9
Nombre de variables utilisées : 9


**CatBoost**

In [26]:
pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 78.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]1/2 [catboost]
Note: you may need to restart the kernel to use updated packages.


In [27]:
from catboost import CatBoostClassifier

In [28]:
# ================================
# CatBoost + Bayésien (SANS SMOTE)
# ================================

# 1) Séparation X / y
X_train_bayes, y_train_bayes = separation(df_train_bayes)
X_test_bayes,  y_test        = separation(df_test_bayes)

# 2) Suppression des identifiants
ID_COLS = ["Transaction_ID", "User_ID"]
X_train_bayes = X_train_bayes.drop(columns=ID_COLS)
X_test_bayes  = X_test_bayes.drop(columns=ID_COLS)

# 3) Détection des variables catégorielles
cat_cols = X_train_bayes.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Variables catégorielles utilisées :", cat_cols)

# 4) Modèle CatBoost
model_cb = CatBoostClassifier(
    iterations=600,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    class_weights=[
        1,  # classe 0
        len(y_train_bayes[y_train_bayes == 0]) / len(y_train_bayes[y_train_bayes == 1])
    ],
    random_seed=42,
    verbose=100
)

# 5) Entraînement
model_cb.fit(
    X_train_bayes,
    y_train_bayes,
    cat_features=cat_cols
)

# 6) Prédictions
y_proba_cb = model_cb.predict_proba(X_test_bayes)[:, 1]
y_pred_cb  = (y_proba_cb >= 0.5).astype(int)


Variables catégorielles utilisées : ['Transaction_Type', 'Device_Used', 'Location', 'Payment_Method']
0:	total: 67.9ms	remaining: 40.7s
100:	total: 881ms	remaining: 4.35s
200:	total: 1.75s	remaining: 3.47s
300:	total: 3s	remaining: 2.98s
400:	total: 4.35s	remaining: 2.16s
500:	total: 5.73s	remaining: 1.13s
599:	total: 7.06s	remaining: 0us


In [29]:
# ================================
# Évaluation CatBoost + Bayésien
# ================================

print("\n=== CatBoost + Bayésien (sans SMOTE) ===")
evaluate_metrics(
    y_true=y_test,
    y_proba=y_proba_cb,
    y_pred=y_pred_cb,
    synthese=False
)



=== CatBoost + Bayésien (sans SMOTE) ===
===== Évaluation du modèle =====
AUC : 0.4875
Seuil optimal (Youden) : 0.5644
--- Au seuil optimal ---
Recall (sensibilité) : 0.0777
Precision : 0.0575
Specificity : 0.9341
F1-score : 0.0661

--- Au seuil 0.5 ---
Accuracy : 0.7001
Recall : 0.2490
Precision : 0.0445
F1-score : 0.0756
